# Entrenamiento v3 — Comida Mexicana (Kaggle)

**Modelo base:** EfficientNet-B0, 87.53% Top-1 (101 clases Food-101)

**Objetivo:** Agregar 20 clases de comida mexicana (121 total)

## Instrucciones previas
1. Crear notebook con **GPU T4** accelerator
2. Activar **Internet** en settings
3. Agregar datasets via "Add Data":
   - `kmader/food41` (Food-101 dataset)
   - Tu dataset con imágenes mexicanas (subido previamente a Kaggle)
4. Actualizar `MEXICAN_FOOD_SLUG` abajo con el slug de tu dataset

In [ ]:
# 1. Verificar GPU
import torch
assert torch.cuda.is_available(), "GPU no detectada. Activa GPU T4 en settings."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# 2. Configurar paths para Kaggle
import os

# Directorio de output (writable)
OUTPUT_DIR = "/kaggle/working"
KAGGLE_OUTPUT = f"{OUTPUT_DIR}/models"
KAGGLE_LOGS = f"{OUTPUT_DIR}/logs"

# Dataset de comida mexicana (cambiar con tu slug)
# Formato: /kaggle/input/<nombre-dataset>/<carpeta>
MEXICAN_FOOD_SLUG = "TU-USUARIO/mexican-food-images"  # <-- CAMBIAR
MEXICAN_FOOD_PATH = f"/kaggle/input/{MEXICAN_FOOD_SLUG}/mexican_food"

# Si no existe el dataset mexicano, usar directorio local
if not os.path.exists(MEXICAN_FOOD_PATH):
    print(f"WARNING: Dataset mexicano no encontrado en {MEXICAN_FOOD_PATH}")
    print("Opciones:")
    print("  1. Sube tus imágenes mexicanas a Kaggle como dataset")
    print("  2. Actualiza MEXICAN_FOOD_SLUG arriba con el slug correcto")
    print("  3. O ejecuta la celda de recolección de imágenes abajo")
    MEXICAN_FOOD_PATH = None

# Crear directorios de output
for d in [OUTPUT_DIR, KAGGLE_OUTPUT, KAGGLE_LOGS]:
    os.makedirs(d, exist_ok=True)

print(f"Output: {KAGGLE_OUTPUT}")
print(f"Logs: {KAGGLE_LOGS}")
if MEXICAN_FOOD_PATH:
    print(f"Dataset mexicano: {MEXICAN_FOOD_PATH}")

In [ ]:
# 3. Clonar repo (requiere Internet ON)
!git clone https://github.com/Fernando-Alvarado-Soria/app-caloriasv2.git
%cd app-caloriasv2

In [ ]:
# 4. Instalar dependencias
# Kaggle ya tiene: torch, torchvision, numpy, pandas, sklearn
# Solo instalamos lo que falta
!pip install duckduckgo_search Pillow pyyaml --quiet

print("Dependencias instaladas")

In [ ]:
# 5. (Opcional) Recolectar imágenes mexicanas si no tienes dataset
# Solo ejecutar si MEXICAN_FOOD_PATH es None

if MEXICAN_FOOD_PATH is None:
    print("Recolectando imágenes mexicanas via DuckDuckGo...")
    print("Esto tarda ~10-15 minutos\n")
    
    from ml.collect_images import collect_all, MEXICAN_CLASSES
    
    stats = collect_all(per_class=150)
    MEXICAN_FOOD_PATH = "ml/data/mexican_food"
    print(f"\nImágenes guardadas en: {MEXICAN_FOOD_PATH}")
else:
    # Verificar imágenes del dataset montado
    import os
    total = 0
    if os.path.exists(MEXICAN_FOOD_PATH):
        for cls in sorted(os.listdir(MEXICAN_FOOD_PATH)):
            cls_dir = os.path.join(MEXICAN_FOOD_PATH, cls)
            if os.path.isdir(cls_dir):
                n = len([f for f in os.listdir(cls_dir) if f.endswith((".jpg",".jpeg",".png"))])
                total += n
                print(f"  {cls:<25} {n} imágenes")
        print(f"\nTotal: {total} imágenes de comida mexicana")
    else:
        print(f"ERROR: {MEXICAN_FOOD_PATH} no existe")

In [ ]:
# 6. Configurar entrenamiento
import ml.config as config

config.BATCH_SIZE = 64
config.NUM_WORKERS = 2
config.DEVICE = "cuda"
config.NUM_EPOCHS = 15

# Fine-tuning completo desde el inicio
config.FREEZE_BACKBONE_EPOCHS = 0
config.UNFREEZE_AFTER = 0

# LR bajo para fine-tuning
config.LEARNING_RATE = 3e-4
config.LR_BACKBONE = 3e-5

# Regularización
config.LABEL_SMOOTHING = 0.1
config.MIXUP_ALPHA = 0.2
config.RANDOM_ERASING_PROB = 0.25
config.EARLY_STOP_PATIENCE = 6
config.CHECKPOINT_EVERY = 5

print("Configuración lista")

# Guardar config como YAML para referencia
import yaml
config_yaml = {
    'version': 'v3_mexican',
    'platform': 'kaggle',
    'batch_size': config.BATCH_SIZE,
    'num_epochs': config.NUM_EPOCHS,
    'learning_rate': config.LEARNING_RATE,
    'lr_backbone': config.LR_BACKBONE,
    'label_smoothing': config.LABEL_SMOOTHING,
    'mixup_alpha': config.MIXUP_ALPHA,
    'random_erasing_prob': config.RANDOM_ERASING_PROB,
    'early_stop_patience': config.EARLY_STOP_PATIENCE,
    'freeze_backbone_epochs': config.FREEZE_BACKBONE_EPOCHS,
    'unfreeze_after': config.UNFREEZE_AFTER,
}
config_path = f'{KAGGLE_LOGS}/config_v3.yaml'
with open(config_path, 'w') as f:
    yaml.dump(config_yaml, f, default_flow_style=False)
print(f"Config guardada: {config_path}")

In [ ]:
# 7. Entrenar con dataset combinado (Food-101 + Mexican Food)
import os
import time
import csv
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models

from ml.config import (
    DATA_DIR, MODELS_DIR, IMAGE_SIZE, NUM_WORKERS,
    RANDOM_CROP_SCALE, COLOR_JITTER, RANDOM_ROTATION,
    RANDOM_ERASING_PROB, MIXUP_ALPHA, LABEL_SMOOTHING,
    LEARNING_RATE, LR_BACKBONE, WEIGHT_DECAY,
    EARLY_STOP_PATIENCE, CHECKPOINT_EVERY,
)
from ml.train import (
    build_transforms, get_device, mixup_data, mixup_criterion,
    evaluate, save_checkpoint,
)
from ml.custom_dataset import CombinedFoodDataset

device = get_device()
print(f"Device: {device}")

# ─── Transforms ───
train_transform, val_transform = build_transforms()

# ─── Dataset combinado ───
print("\nCargando dataset combinado...")
train_dataset = CombinedFoodDataset(
    root=DATA_DIR, split='train', transform=train_transform,
    download=True, custom_dir=MEXICAN_FOOD_PATH,
)
val_dataset = CombinedFoodDataset(
    root=DATA_DIR, split='test', transform=val_transform,
    download=True, custom_dir=MEXICAN_FOOD_PATH,
)

NUM_CLASSES = train_dataset.num_classes
print(f"\nTotal clases: {NUM_CLASSES}")
print(f"  Food-101: {len(train_dataset.base_classes)}")
print(f"  Mexicanas: {len(train_dataset.custom_classes)}")

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# ─── Modelo: cargar v2 desde best_model.pt ───
print("\nCargando modelo v2 (87.53%)...")
from ml.train import build_model

# Buscar best_model.pt en varias ubicaciones
best_model_paths = [
    'ml/models/best_model.pt',
    '/kaggle/input/<tu-modelo>/best_model.pt',  # Si lo subes como dataset
]

model = build_model(device)
model_loaded = False

for bp in best_model_paths:
    if os.path.exists(bp):
        print(f"Cargando checkpoint desde: {bp}")
        checkpoint = torch.load(bp, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Pesos v2 cargados (época {checkpoint.get('epoch', '?')}, "
              f"val_acc={checkpoint.get('val_acc', 0):.1f}%)")
        model_loaded = True
        break

if not model_loaded:
    print("WARNING: No se encontró best_model. Usando pesos de ImageNet.")

# Expandir clasificador de 101 → NUM_CLASSES
if NUM_CLASSES > 101:
    old_classifier = model.classifier
    in_features = old_classifier[1].in_features
    
    new_classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, NUM_CLASSES),
    ).to(device)
    
    with torch.no_grad():
        new_classifier[1].weight[:101] = old_classifier[1].weight
        new_classifier[1].bias[:101] = old_classifier[1].bias
    
    model.classifier = new_classifier
    print(f"Clasificador expandido: 101 → {NUM_CLASSES} clases")

# ─── Entrenamiento con Logging ───
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
best_val_acc = 0.0
patience_counter = 0

# Guardar lista de clases
os.makedirs(MODELS_DIR, exist_ok=True)
classes_path = os.path.join(MODELS_DIR, 'classes.txt')
with open(classes_path, 'w') as f:
    for cls in train_dataset.classes:
        f.write(cls + '\n')
print(f"Clases guardadas: {classes_path} ({NUM_CLASSES} clases)")

# CSV de logging
log_csv = f'{KAGGLE_LOGS}/training_v3.csv'
log_header = ['epoch', 'train_loss', 'train_acc', 'val_loss', 'val_acc', 'val_top5', 'lr', 'time_s']
with open(log_csv, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(log_header)
print(f"Logging en: {log_csv}")

print(f"\n{'='*60}")
print(f"ENTRENANDO: {config.NUM_EPOCHS} épocas, {NUM_CLASSES} clases")
print(f"{'='*60}\n")

for epoch in range(config.NUM_EPOCHS):
    t0 = time.time()
    model.train()
    
    classifier_params = []
    backbone_params = []
    for name, param in model.named_parameters():
        if 'classifier' in name:
            classifier_params.append(param)
        else:
            backbone_params.append(param)
    
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': LR_BACKBONE},
        {'params': classifier_params, 'lr': LEARNING_RATE},
    ], weight_decay=WEIGHT_DECAY)
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=len(train_loader), eta_min=1e-6
    )
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        if MIXUP_ALPHA > 0:
            images, targets_a, targets_b, lam = mixup_data(images, labels, MIXUP_ALPHA)
            optimizer.zero_grad()
            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        else:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        if (batch_idx + 1) % 100 == 0:
            print(f"  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} | "
                  f"Loss: {loss.item():.4f} | Acc: {100.*correct/total:.1f}%")
    
    train_loss = running_loss / total
    train_acc = 100. * correct / total
    
    val_loss, val_acc, val_top5 = evaluate(model, val_loader, criterion, device)
    
    elapsed = time.time() - t0
    current_lr = scheduler.get_last_lr()[0]
    
    print(f"\nÉpoca {epoch+1}/{config.NUM_EPOCHS} ({elapsed:.0f}s)")
    print(f"  Train — Loss: {train_loss:.4f} | Acc: {train_acc:.1f}%")
    print(f"  Val   — Loss: {val_loss:.4f} | Top-1: {val_acc:.1f}% | Top-5: {val_top5:.1f}%")
    
    # Log a CSV
    with open(log_csv, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            epoch+1, f'{train_loss:.4f}', f'{train_acc:.1f}',
            f'{val_loss:.4f}', f'{val_acc:.1f}', f'{val_top5:.1f}',
            f'{current_lr:.6f}', f'{elapsed:.0f}'
        ])
    
    # Checkpoint a /kaggle/working/
    if (epoch + 1) % CHECKPOINT_EVERY == 0:
        cp_path = os.path.join(MODELS_DIR, f'checkpoint_epoch{epoch+1}.pt')
        save_checkpoint(model, optimizer, epoch, val_acc, cp_path)
        import shutil
        shutil.copy2(cp_path, f'{KAGGLE_OUTPUT}/checkpoint_epoch{epoch+1}.pt')
    
    # Mejor modelo
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        best_path = os.path.join(MODELS_DIR, 'best_model.pt')
        save_checkpoint(model, optimizer, epoch, val_acc, best_path)
        import shutil
        shutil.copy2(best_path, f'{KAGGLE_OUTPUT}/best_model.pt')
        print(f"  ★ Nuevo mejor modelo: {val_acc:.1f}% (guardado en working)")
    else:
        patience_counter += 1
    
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping: sin mejora en {EARLY_STOP_PATIENCE} épocas.")
        break
    print()

print(f"\nEntrenamiento completado. Mejor Val Top-1: {best_val_acc:.1f}%")
print(f"Logs: {log_csv}")
print(f"Modelos: {KAGGLE_OUTPUT}")

In [ ]:
# 8. Exportar modelo con nuevas clases
import json
import shutil

# Cargar mejor modelo
best_path = os.path.join(MODELS_DIR, 'best_model.pt')
if not os.path.exists(best_path):
    best_path = f'{KAGGLE_OUTPUT}/best_model.pt'

checkpoint = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

val_acc = checkpoint.get('val_acc', 0)
epoch = checkpoint.get('epoch', '?')
print(f"Mejor modelo: época {epoch}, val_acc={val_acc:.1f}%")

# Exportar TorchScript
export_dir = os.path.join(MODELS_DIR, 'export')
os.makedirs(export_dir, exist_ok=True)

example_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
scripted_model = torch.jit.trace(model, example_input)
scripted_path = os.path.join(export_dir, 'model_scripted.pt')
scripted_model.save(scripted_path)
print(f"TorchScript: {scripted_path}")

# State dict
inference_path = os.path.join(export_dir, 'model_inference.pt')
torch.save(model.state_dict(), inference_path)

# Metadata
classes = [line.strip() for line in open(classes_path)]
metadata = {
    'model_name': 'efficientnet_b0',
    'num_classes': NUM_CLASSES,
    'image_size': IMAGE_SIZE,
    'val_accuracy_top1': round(val_acc, 2),
    'epoch': epoch,
    'classes': classes,
    'normalize': {
        'mean': [0.485, 0.456, 0.406],
        'std': [0.229, 0.224, 0.225],
    },
    'custom_classes': train_dataset.custom_classes,
}
meta_path = os.path.join(export_dir, 'metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata: {meta_path}")

# Copiar todo a /kaggle/working/ para descarga
drive_export = f'{KAGGLE_OUTPUT}/export'
if os.path.exists(drive_export):
    shutil.rmtree(drive_export)
shutil.copytree(export_dir, drive_export)
print(f"\nExport copiado a: {drive_export}")

# Copiar classes.txt
shutil.copy2(classes_path, f'{KAGGLE_OUTPUT}/classes.txt')

print(f"\n{'='*60}")
print("ARCHIVOS LISTOS PARA DESCARGAR")
print(f"{'='*60}")
print(f"\nVe a la pestaña 'Output' del kernel y descarga:")
print(f"  - best_model.pt")
print(f"  - export/model_scripted.pt")
print(f"  - export/metadata.json")
print(f"  - classes.txt")
print(f"  - training_v3.csv")
print(f"\nClases totales: {NUM_CLASSES} (101 Food-101 + {len(train_dataset.custom_classes)} mexicanas)")

In [ ]:
# 9. Resumen final
print("\n" + "="*60)
print("RESUMEN DE ENTRENAMIENTO v3")
print("="*60)
print(f"\nPlataforma: Kaggle (GPU T4)")
print(f"Modelo: EfficientNet-B0")
print(f"Clases: {NUM_CLASSES} (101 Food-101 + {len(train_dataset.custom_classes)} mexicanas)")
print(f"Mejor Val Top-1: {best_val_acc:.1f}%")
print(f"Épocas entrenadas: {epoch+1}")
print(f"\nArchivos generados en /kaggle/working/models/:")
print(f"  - best_model.pt (checkpoint completo)")
print(f"  - export/model_scripted.pt (TorchScript)")
print(f"  - export/metadata.json (clases + config)")
print(f"  - classes.txt (lista de clases)")
print(f"\nPróximos pasos:")
print(f"  1. Descargar archivos desde la pestaña Output")
print(f"  2. Copiar a ml/models/export/ en el repo")
print(f"  3. Push a GitHub y deployar a Railway")